# Predicting the Sale Price of Bulldozers using Machine Learning

## 1.Problem definition
> How well can we predict the future sale price of a bulldozer, given its characteristics previous examples of how much similar bulldozers have been sold for?

## 2.Data
The data is downloaded from the Kaggle Bluebook for Bulldozers competition:
https://www.kaggle.com/competitions/bluebook-for-bulldozers/data

*** Note: Place the files in a `data/` folder at the root of the project. ***

There are 3 main datasets:

* Train.csv is the training set, which contains data through the end of 2011.
* Valid.csv is the validation set, which contains data from January 1, 2012 - April 30, 2012 You make predictions on this set throughout the majority of the competition. Your score on this set is used to create the public leaderboard.
* Test.csv is the test set, which won't be released until the last week of the competition. It contains data from May 1, 2012 - November 2012. Your score on the test set determines your final rank for the competition.

## 3.Evaluation
The evaluation metric for this competition is the RMSLE (root mean squared log error) between the actual and predicted auction prices.

## 4.Features
Kaggle provides a data dictionary detailing all of te features of the dataset. You can view this data dictionary on the file: 

```
bulldozer-price-prediction/
└── data/
    └── Data Dictionary.xlsx   # This file
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import sklearn

# Plot appear inside the notebook
%matplotlib inline

In [ ]:
# Importing training and validation datasets
df = pd.read_csv("data/TrainAndValid.csv", low_memory=False)
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df.columns

In [ ]:
fig, ax = plt.subplots()
fig.patch.set_facecolor('white')  
ax.set_facecolor('white')       
ax.grid(False)  

ax.scatter(df["saledate"][:1000], df["SalePrice"][:1000])
plt.show()

In [ ]:
df.saledate[:1000]

In [ ]:
df.saledate.dtypes

In [ ]:
df.SalePrice.plot.hist()

In [ ]:
# Parsing dates
df = pd.read_csv("data/TrainAndValid.csv", low_memory=False, parse_dates=["saledate"])


In [ ]:
df.saledate.dtypes

In [ ]:
df.saledate[:1000]

In [ ]:
fig, ax = plt.subplots()
ax.scatter(df["saledate"][:1000], df["SalePrice"][:1000])
plt.show()

In [ ]:
df.head()

In [ ]:
df.head().T

In [ ]:
# sorting by saledate in date order
df.sort_values(by=["saledate"], inplace=True)
df.saledate.head(20)

In [ ]:
# make a copy
df_temp = df.copy()

In [ ]:
df_temp.saledate.head(20)

In [ ]:
df_temp['saleYear'] = df_temp["saledate"].dt.year
df_temp['saleMonth'] = df_temp["saledate"].dt.month
df_temp['saleDay'] = df_temp["saledate"].dt.day
df_temp['saleDayOfWeek'] = df_temp["saledate"].dt.day_of_week
df_temp['saleDayOfYear'] = df_temp["saledate"].dt.day_of_year

In [ ]:
df_temp.drop('saledate', axis=1, inplace=True)

### Convert string to categories

In [ ]:
df_temp.head().T

In [ ]:
str_columns = df_temp.select_dtypes(include=["object", "string"]).columns

for col in str_columns:
    df_temp[col] = df_temp[col].astype("category").cat.as_ordered()

df_temp.info()

In [ ]:
df_temp["state"].cat.categories

In [ ]:
df_temp["state"].cat.codes

### Check missing data

In [ ]:
# Export current tmp dataframe
df_temp.to_csv("data/train_tmp.csv", index=False)

# import data tmp
df_tmp = pd.read_csv("data/train_tmp.csv", low_memory=False)
df_temp.head().T

#### fill numeric values

In [ ]:
numeric_columns = df_temp.select_dtypes(include=["number"]).columns

for column in numeric_columns:
    missing_mask = df_temp[column].isnull()
    if missing_mask.any():
        df_temp[column + "_is_missing"] = missing_mask
        df_temp[column] = df_temp[column].fillna(df_temp[column].median())

for column in numeric_columns:
    if df_temp[column].isna().sum():
        print(f"{column} still has missing values")


In [ ]:
df_temp["auctioneerID_is_missing"].value_counts()

#### fill categorical values

In [ ]:
print(df_temp["UsageBand"].cat.codes)

In [ ]:
category_columns = df_temp.select_dtypes(include=["category"]).columns

for column in category_columns:
    missing_mask = df_temp[column].isnull()
    if missing_mask.any():
        df_temp[column + "_is_missing"] = missing_mask
    df_temp[column] = df_temp[column].cat.codes + 1

for column in category_columns:
    if df_temp[column].isna().sum():
        print(f"{column} still has missing values")

In [ ]:
df_temp.select_dtypes(include=["category"])

In [ ]:
df_temp.info()

In [ ]:
df_temp.isna().sum()[:20]

# Modeling

In [ ]:
%%time

from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_jobs=-1, random_state=42)
model.fit(df_temp.drop("SalePrice", axis=1), df_temp["SalePrice"])